# Get NISAR GSLC Image Chips per ISMN Station

<br>

For each ISMN soil moisture station in `ismn_data/<AOI>/stations.geojson`, **processed
independently of the others**, this notebook saves a small pixel window of NISAR L-band
(frequency A) **GSLC** (Geocoded Single-Look Complex) data around the station, for every
acquisition covering it -- one NetCDF file per acquisition per station.

Each saved file contains, for whichever polarizations the granule has (HH/HV/VH/VV):
the complex SLC window split into `<pol>_real`/`<pol>_imag` (NetCDF has no native complex
type), plus per-pixel geometry cubes covering the same window
(`incidence_angle_deg`, `los_unit_vector_x`/`y`, `local_incidence_angle_deg`) and the
`mask` layer if present, with station/orbit/acquisition metadata as file attributes.
Every polarization present is saved here so nothing is discarded at extraction time,
but only **HH** is actually used downstream: `02b_calc_nisar_coherence_closure.ipynb`
computes both `coherence_HH` and `coherence_HV`, but `03a_estimate_sm_nisar.ipynb`'s
inversion only reads HH by default (`POLARIZATIONS = ("HH",)`) -- HV is
computed but unused, and this granule's own `listOfPolarizations` (Section 2 below)
confirms NISAR's typical mode here is HH/HV, not full quad-pol (no VH/VV).
Filenames encode the station name, acquisition datetime, track number, orbit direction,
and the full product native-id, e.g.:

`{station_name}_{YYYYMMDDTHHMMSS}_track{NNN}_{ASC|DESC}_{native_id}.nc`

under `nisar_data/<AOI>_GSLC_windows/`.

It streams each granule remotely with `fsspec` + `h5py` -- no full-granule downloads.

<hr>

## Overview

1. Load station locations
2. Sanity-check GSLC file structure against a real granule
3. Compute local terrain slope/aspect per station from Copernicus GLO-30 (helper, reused from `util/nisar_station_timeseries.py`)
4. Process each station independently: search, extract window, save
5. (Optional) Visualize combined granule coverage across all stations

<hr>

In [1]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timezone

from ipyleaflet import CircleMarker
import earthaccess

from util.aoi_coverage import plot_aoi_coverage
from util.nisar_station_timeseries import compute_station_terrain, get_fsspec_https_session_with_timeout
from util.nisar_gslc_windows import (
    GSLC_SHORT_NAME,
    DEFAULT_WINDOW_FOOTPRINT_M,
    inspect_gslc_structure,
    process_station_gslc,
)

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Load station locations

Same station list and `TEMPORAL` window convention used elsewhere in this
pipeline. `elevation_m` here is ISMN metadata kept only for reference --
`dem_elevation_m` (Section 3) is what's actually used for height
interpolation.

`WINDOW_FOOTPRINT_M` is the target window size in meters on the ground, not
pixels -- matches the HyP3 Sentinel-1 burst product's pixel size (80m) so
both sensors' windows cover comparable ground footprints. GSLC's grid is
anisotropic, so x/y pixel counts are derived per granule from this target
rather than a fixed pixel count (see `extract_and_save_gslc_window`).

In [2]:
# stations.geojson's AOI subfolder -- this example run uses Alaska.
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = [
    {
        "name": feat["properties"]["station_val"],
        "lon": feat["geometry"]["coordinates"][0],
        "lat": feat["geometry"]["coordinates"][1],
        "elevation_m": float(feat["properties"]["elevation_val"]),
    }
    for feat in stations_geojson["features"]
]

TEMPORAL = ("2026-06-15", datetime.now(timezone.utc).strftime("%Y-%m-%d"))
WINDOW_FOOTPRINT_M = DEFAULT_WINDOW_FOOTPRINT_M  # meters on the ground; see util/nisar_gslc_windows.py

print(f"Loaded {len(stations)} stations")
stations[:3]

Loaded 17 stations


[{'name': 'MooseInc', 'lon': -151.3872, 'lat': 59.68255, 'elevation_m': 37.0},
 {'name': 'AtigunPass',
  'lon': -149.47817,
  'lat': 68.12983,
  'elevation_m': 1457.0},
 {'name': 'Coldfoot', 'lon': -150.183, 'lat': 67.25333, 'elevation_m': 326.0}]

## 2. Sanity-check GSLC file structure against a real granule

Opens one real GSLC granule (the first candidate found for the first station) and prints
its `frequencyA`/`radarGrid`/`identification` layout via `inspect_gslc_structure()`.

In [ ]:
earthaccess.login()

fs = get_fsspec_https_session_with_timeout()
fsspec_config = {
    "cache_type": "background",
    "block_size": 8 * 1024 * 1024,  # 8 MB
}

_sanity_results = earthaccess.search_data(
    short_name=GSLC_SHORT_NAME,
    point=(stations[0]["lon"], stations[0]["lat"]),
    temporal=TEMPORAL,
)
print(f"Found {len(_sanity_results)} candidate GSLC granule(s) for station '{stations[0]['name']}'")

if _sanity_results:
    inspect_gslc_structure(_sanity_results[0], fs, fsspec_config)
else:
    print("No candidates found — check GSLC_SHORT_NAME in util/nisar_gslc_windows.py against a live CMR search.")

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Found 19 candidate GSLC granule(s) for station 'MooseInc'
native-id: NISAR_L2_PR_GSLC_023_078_A_032_4005_DHDH_A_20260619T143045_20260619T143121_P05023_N_F_J_001

Top-level groups: ['GSLC', 'identification']

/science/LSAR/GSLC/grids/frequencyA contents:
  HH: shape=(76320, 76464) dtype=complex64
  HV: shape=(76320, 76464) dtype=complex64
  azimuthBandwidth: shape=() dtype=float64
  centerFrequency: shape=() dtype=float64
  inputDataExceptionMask: shape=(76320, 76464) dtype=uint8
  listOfPolarizations: shape=(2,) dtype=|S2
  mask: shape=(76320, 76464) dtype=uint8
  numberOfSubSwaths: shape=() dtype=uint8
  projection: shape=() dtype=uint32
  rangeBandwidth: shape=() dtype=float64
  slantRangeSpacing: shape=() dtype=float64
  xCoordinateSpacing: shape=() dtype=float64
  xCoordinates: shape=(76464,) dtype=float64
  yCoordinateSpacing: shape=() dtype=float64
  yCoordinates: shape=(76320,) dtype=float64
  zeroDopplerTimeSpacing: shape=() dtype=float64
  projection raw value: 32605, attrs: {

## 3. Compute local terrain slope/aspect per station from Copernicus GLO-30

Reuses `compute_station_terrain()` from `util/nisar_station_timeseries.py`
directly. Terrain slope/aspect isn't recomputed for all pixels, since it varies negligibly across a chip this size.

In [4]:
compute_station_terrain(stations)

Computing local terrain slope/aspect from Copernicus GLO-30 for each station...
Done. 15 unique DEM tiles used for 17 stations.
Slope range across stations: 0.80 - 9.41 deg (median 2.59 deg)


## 4. Process each station independently: search, extract window, save

Implemented by `process_station_gslc()` in
[`util/nisar_gslc_windows.py`](../util/nisar_gslc_windows.py) -- see that
module's docstrings for the full search/dedup/extract steps and saved-file
contents. Briefly: search, dedup, filter by track-acquisition count, skip
granules already saved, then for each new granule extract a window around
the station's nearest pixel (HH/HV/VH/VV present, `mask`, and a per-pixel
geometry cube and save it as its own NetCDF file.

In [5]:
# Output directory, named for this example run's AOI (Alaska).
output_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_windows"
output_dir.mkdir(parents=True, exist_ok=True)

all_gpolygons = {}  # native-id -> GPolygon, collected for the optional overview map below

for station in stations:
    process_station_gslc(
        station, output_dir, TEMPORAL, fsspec_config,
        window_footprint_m=WINDOW_FOOTPRINT_M, all_gpolygons=all_gpolygons,
    )

MooseInc: 19 granule(s) after version dedup and track filtering — 0 saved, 19 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


AtigunPass: 19 granule(s) after version dedup and track filtering — 0 saved, 19 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Coldfoot: 20 granule(s) after version dedup and track filtering — 0 saved, 20 already saved, 0 skipped (no usable data for this station), 0 timed out


...

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


TurnagainPass: 24 granule(s) after version dedup and track filtering — 0 saved, 24 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


UpperNomeCreek: 38 granule(s) after version dedup and track filtering — 0 saved, 38 already saved, 0 skipped (no usable data for this station), 0 timed out
Kenai-29-ENE: 24 granule(s) after version dedup and track filtering — 0 saved, 24 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


## 5. (Optional) Visualize combined granule coverage across all stations

A single overview map built from every unique granule footprint encountered across all per-station searches above, reusing `plot_aoi_coverage` from `util/aoi_coverage.py`.

In [6]:
lons = [s["lon"] for s in stations]
lats = [s["lat"] for s in stations]
pad = 0.05

aoi_wkt = (
    "POLYGON(("
    f"{min(lons) - pad} {min(lats) - pad}, {max(lons) + pad} {min(lats) - pad}, "
    f"{max(lons) + pad} {max(lats) + pad}, {min(lons) - pad} {max(lats) + pad}, "
    f"{min(lons) - pad} {min(lats) - pad}"
    "))"
)

m = plot_aoi_coverage(list(all_gpolygons.values()), aoi_wkt, "GSLC")

for s in stations:
    m.add(CircleMarker(
        location=(s["lat"], s["lon"]),
        radius=4,
        color="red",
        fill_color="red",
        fill_opacity=0.9,
    ))

m

Map(center=[41.570265, 1.1064349999999998], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

<hr>

## Summary

You now have one NetCDF file per GSLC acquisition per station in
`nisar_data/alaska_GSLC_windows/`, named
`{station_name}_{datetime}_track{NNN}_{ASC|DESC}_{native_id}.nc` -- see
`util/nisar_gslc_windows.py`'s docstrings for the full contents (complex
polarization windows, mask, per-pixel geometry cube, file attrs).

Reload with `xr.open_dataset(path)`; reconstruct a polarization's complex
values with `ds["HH_real"].values + 1j * ds["HH_imag"].values`.

Re-running after `TEMPORAL`'s end date moves forward only fetches
newly-acquired granules -- already-saved ones are skipped, no network calls
needed.

<hr>